# Pobreza multidimensional en Colombia: factores asociados según la ECV 2023
**APES – Primera entrega** · Jeisson Sánchez y Juan Bogotá

Fuente: Encuesta Nacional de Calidad de Vida (ECV) 2023, DANE. Unidad de análisis: hogar y persona (llave `DIRECTORIO` + `SECUENCIA_P` [+ `ORDEN`]).

## 0. SETUP

In [ ]:
%pip install -q openpyxl pyarrow

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(42)

# Ruta base: raíz del repo (el notebook vive en notebooks/)
BASE_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))

RAW_CSV   = os.path.join(BASE_PATH, "data", "raw", "csv")
PROCESSED = os.path.join(BASE_PATH, "data", "processed")
FIGURES   = os.path.join(BASE_PATH, "figures")
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(FIGURES, exist_ok=True)

print("BASE_PATH:", BASE_PATH)
print("CSV encontrados:", sorted(os.listdir(RAW_CSV)))

## 1. CARGA DE DATOS

In [ ]:
ARCHIVOS = {
    "personas":        "personas.csv",
    "vivienda":        "vivienda.csv",
    "servicios":       "servicios.csv",
    "condiciones":     "condiciones.csv",
    "educacion":       "educacion.csv",
    "fuerza_trabajo":  "fuerza_trabajo.csv",
    "salud":           "salud.csv",
    "diseno_muestra":  "diseno_muestra.csv",
    "trabajo_infantil":"trabajo_infantil.csv",
    "primera_infancia":"primera_infancia.csv",
    "tic":             "tic.csv",
}

def leer_csv(ruta):
    """Lee un CSV del DANE probando separador y codificación."""
    with open(ruta, "rb") as f:
        primera = f.readline()
    for enc in ("utf-8", "latin-1"):
        try:
            linea = primera.decode(enc)
            break
        except UnicodeDecodeError:
            continue
    sep = ";" if linea.count(";") > linea.count(",") else ","
    return pd.read_csv(ruta, sep=sep, encoding=enc, low_memory=False)

dfs = {}
for nombre, archivo in ARCHIVOS.items():
    dfs[nombre] = leer_csv(os.path.join(RAW_CSV, archivo))
    dfs[nombre].columns = dfs[nombre].columns.str.strip().str.upper()

# Resumen de dimensiones
resumen = pd.DataFrame({
    "filas":    {k: v.shape[0] for k, v in dfs.items()},
    "columnas": {k: v.shape[1] for k, v in dfs.items()},
})
resumen

In [ ]:
# Verificación de llaves y tamaños esperados (86.405 hogares, 240.212 personas)
LLAVE_HOGAR = ["DIRECTORIO", "SECUENCIA_P"]
LLAVE_PERSONA = LLAVE_HOGAR + ["ORDEN"]

for nombre, df in dfs.items():
    tiene_hogar   = all(c in df.columns for c in LLAVE_HOGAR)
    tiene_persona = all(c in df.columns for c in LLAVE_PERSONA)
    print(f"{nombre:18s} llave hogar: {tiene_hogar!s:5} | llave persona: {tiene_persona!s:5}")

n_hogares  = dfs["personas"][LLAVE_HOGAR].drop_duplicates().shape[0]
n_personas = dfs["personas"].shape[0]
print(f"\nHogares en 'personas':  {n_hogares:,}  (esperado 86.405)")
print(f"Personas en 'personas': {n_personas:,}  (esperado 240.212)")

In [ ]:
# Vista rápida de cada módulo (solo las primeras 10 columnas)
for nombre, df in dfs.items():
    print(f"\n=== {nombre} {df.shape} ===")
    display(df.head(3).iloc[:, :10])

**Conclusión.** Los 11 módulos cargan y las llaves están en todos. `personas` tiene 86.405 hogares y 240.212 personas, igual que la ficha técnica.

## 2. INTEGRACIÓN
### 2.1 Duplicados por llave en cada módulo

In [ ]:
filas = []
for nombre, df in dfs.items():
    filas.append({
        "modulo": nombre,
        "filas": len(df),
        "dup_llave_hogar": df.duplicated(LLAVE_HOGAR).sum(),
        "dup_llave_persona": df.duplicated(LLAVE_PERSONA).sum(),
        "ORDEN_unicos": df["ORDEN"].nunique(),
    })
pd.DataFrame(filas).set_index("modulo")

**Conclusión.** En ningún módulo se repite la llave de persona (`DIRECTORIO` + `SECUENCIA_P` + `ORDEN`). `servicios` y `condiciones` tienen 342 llaves de hogar repetidas y `vivienda` tiene una fila por vivienda (86.063), así que en esos módulos la llave de hogar no es la misma que en `personas`.

### 2.2 ¿Cuántos hogares de 'personas' no aparecen en cada módulo de hogar?

In [ ]:
hogares = dfs["personas"][LLAVE_HOGAR].drop_duplicates()
for m in ["vivienda", "servicios", "condiciones"]:
    cruce = hogares.merge(dfs[m][LLAVE_HOGAR].drop_duplicates(), on=LLAVE_HOGAR, how="left", indicator=True)
    print(f"{m:12s} hogares sin registro: {(cruce['_merge'] == 'left_only').sum():,}")

# Alternativa: ¿vivienda se une solo por DIRECTORIO?
sin_dir = hogares[~hogares["DIRECTORIO"].isin(dfs["vivienda"]["DIRECTORIO"])]
print(f"\nHogares cuyo DIRECTORIO no está en vivienda: {len(sin_dir):,}")

**Conclusión.** Con `DIRECTORIO` + `SECUENCIA_P`, 342 hogares quedan sin registro en `vivienda`, `servicios` y `condiciones`. Todos los `DIRECTORIO` sí están en `vivienda`, por lo que `vivienda` se une solo por `DIRECTORIO`.

### 2.3 Buscar variables clave en todos los módulos

In [ ]:
palabras = ["IPM", "POBREZ", "INGRES", "I_HOGAR", "PERCAPITA", "FEX", "FACTOR",
            "CLASE", "REGION", "DOMINIO", "DEPARTAMENTO", "P6040", "P6020", "P6050", "P6008"]

for p in palabras:
    hallazgos = {m: [c for c in df.columns if p in c] for m, df in dfs.items()}
    hallazgos = {m: c for m, c in hallazgos.items() if c}
    print(f"\n[{p}]")
    for m, cols in hallazgos.items():
        print(f"   {m}: {cols[:8]}")

**Conclusión.** El microdato no trae el IPM ni una variable de pobreza: hay que **construirlo** (Sección 4). El ingreso (`I_HOGAR`, `PERCAPITA`) está en `servicios`; el factor de expansión `FEX_C` está en todos los módulos; `CLASE`, `REGION` y `P1_DEPARTAMENTO` están en `vivienda`; edad (`P6040`) y sexo (`P6020`) están en `personas`.

### 2.4 Bases integradas (personas y hogares)

In [ ]:
def unir(base, df, llave, how="left"):
    """Une df a base, agregando solo columnas que base aún no tiene."""
    nuevas = [c for c in df.columns if c not in base.columns or c in llave]
    return base.merge(df[nuevas].drop_duplicates(llave), on=llave, how=how)



df_personas = dfs["personas"].copy()
for m in ["salud", "diseno_muestra", "educacion", "fuerza_trabajo", "tic",
          "trabajo_infantil", "primera_infancia"]:
    df_personas = unir(df_personas, dfs[m], LLAVE_PERSONA)
assert len(df_personas) == 240212, "Se perdieron o duplicaron personas"


def a_llave_hogar(df):
    return df.drop(columns=["SECUENCIA_P"]).rename(columns={"ORDEN": "SECUENCIA_P"})

df_hogares = hogares.copy()
df_hogares = unir(df_hogares, a_llave_hogar(dfs["servicios"]), LLAVE_HOGAR)
df_hogares = unir(df_hogares, a_llave_hogar(dfs["condiciones"]), LLAVE_HOGAR)

# vivienda: una fila por vivienda, se une solo por DIRECTORIO
viv = dfs["vivienda"].drop(columns=["SECUENCIA_P", "ORDEN"], errors="ignore")
df_hogares = unir(df_hogares, viv, ["DIRECTORIO"])

# Tamaño del hogar
tam = df_personas.groupby(LLAVE_HOGAR).size().rename("TAM_HOGAR").reset_index()
df_hogares = df_hogares.merge(tam, on=LLAVE_HOGAR, how="left")
assert len(df_hogares) == 86405, "Se perdieron o duplicaron hogares"

print("df_personas:", df_personas.shape)
print("df_hogares: ", df_hogares.shape)
for c in ["I_HOGAR", "PERCAPITA", "CLASE", "TAM_HOGAR"]:
    print(f"{c} nulo:", df_hogares[c].isna().sum())
print("TAM_HOGAR != CANT_PERSONAS_HOGAR:",
      (df_hogares["TAM_HOGAR"] != df_hogares["CANT_PERSONAS_HOGAR"]).sum())

**Conclusión.** `df_personas` queda con 240.212 filas y `df_hogares` con 86.405. `I_HOGAR`, `PERCAPITA`, `CLASE` y `TAM_HOGAR` no tienen nulos, y el tamaño calculado coincide con `CANT_PERSONAS_HOGAR`. Se conservan todas las columnas por ahora; en la Sección 4.2 se guardan solo las que se usan.

### 2.5 Llave de hogar en servicios y condiciones

In [ ]:
cols_id = lambda df: [c for c in df.columns if "SECUENCIA" in c or c in ("DIRECTORIO", "ORDEN")]
print("servicios:", cols_id(dfs["servicios"]))
print("personas: ", cols_id(dfs["personas"]))
print("personas SECUENCIA_P:", sorted(dfs["personas"]["SECUENCIA_P"].unique())[:10])
print("servicios SECUENCIA_P:", sorted(dfs["servicios"]["SECUENCIA_P"].unique())[:10])
print("servicios ORDEN:", sorted(dfs["servicios"]["ORDEN"].unique()))

# Hipótesis: en servicios/condiciones el hogar es DIRECTORIO + ORDEN
for m in ["servicios", "condiciones"]:
    der = dfs[m][["DIRECTORIO", "ORDEN"]].rename(columns={"ORDEN": "SECUENCIA_P"})
    c = hogares.merge(der, on=LLAVE_HOGAR, how="left", indicator=True)
    print(f"{m}: hogares sin registro con DIRECTORIO+ORDEN = {(c['_merge'] == 'left_only').sum():,}")

**Conclusión.** En `servicios`, `condiciones` y `vivienda`, `SECUENCIA_P` vale siempre 1 y **`ORDEN` es el número de hogar** dentro de la vivienda. Con `DIRECTORIO` + `ORDEN` los 342 hogares faltantes aparecen (0 sin registro). Por eso esos módulos se unen con `a_llave_hogar()`.

### 2.6 Parentesco, tamaño del hogar y estrato

In [ ]:
# 2.6 Parentesco, tamaño del hogar, estrato
for pal in ["PARENT", "JEFE", "P6051", "CANT", "PERSONAS", "ESTRATO", "ZONA", "AREA"]:
    print(f"\n[{pal}]")
    for m, df in dfs.items():
        cols = [c for c in df.columns if pal in c]
        if cols:
            print(f"   {m}: {cols[:8]}")

# Diccionario de datos (primera hoja)
dic = pd.read_excel(os.path.join(BASE_PATH, "data", "docs", "Diccionario de datos_ECV 2023.xlsx"), sheet_name=None)
dd = dic[list(dic.keys())[0]].iloc[:, [0, 3, 9, 10]].copy()
dd.columns = ["modulo", "variable", "pregunta", "categorias"]
dd = dd.dropna(subset=["variable"]).reset_index(drop=True)

def buscar_dic(palabra, n=10):
    """Busca una palabra en las preguntas y categorías del diccionario."""
    pal = palabra.lower()
    m = (dd["pregunta"].astype(str).str.lower().str.contains(pal, regex=False)
         | dd["categorias"].astype(str).str.lower().str.contains(pal, regex=False))
    print(f"\n[{palabra}]")
    for _, f in dd[m].head(n).iterrows():
        print(f"  {f['variable']:10s} | {str(f['modulo'])[:24]:24s} | {str(f['pregunta'])[:95]}")

def ver(var):
    """Muestra pregunta y categorías de una variable."""
    f = dd[dd["variable"] == var].iloc[0]
    print(f"{var} ({f['modulo']})\n{f['pregunta']}\n{f['categorias']}\n")

ver("P6051")
jefes_chk = df_personas[df_personas["P6051"] == 1].groupby(LLAVE_HOGAR).size()
print("Hogares con exactamente 1 jefe:", (jefes_chk == 1).sum(), "de", df_hogares.shape[0])

**Conclusión.** El parentesco es `P6051` (1 = jefe/a del hogar) y el tamaño del hogar también viene en `CANT_PERSONAS_HOGAR`. `ESTRATO2020` es un código de diseño muestral, **no** el estrato socioeconómico; el estrato (1 a 6) es `P8520S1A1` en `vivienda`.

## 3. LIMPIEZA Y VARIABLES DERIVADAS

In [ ]:
# 3. LIMPIEZA Y VARIABLES DERIVADAS
p = df_personas

# 3.1 Años de educación aprobados (persona)
def anos_no_estudiante(n, g):
    g = g.fillna(0)
    return np.select(
        [n.isin([1, 2]), n == 3, n == 4, n == 5, n >= 6],
        [0, g.clip(0, 5), g.clip(5, 9), g.clip(9, 11), 11 + g],
        default=np.nan)

def anos_estudiante(n, g, sup):
    """Años aprobados = grado que cursa - 1 (metodología DANE)."""
    g = g.fillna(1)
    return np.select(
        [n == 1, n == 2, n == 3, n == 4, n >= 5],
        [0,
         (g - 1).clip(0, 4),
         (np.where(g >= 6, g, 5 + g) - 1).clip(5, 8),
         (np.where(g >= 10, g, 9 + g) - 1).clip(9, 11),
         11 + sup.fillna(0)],
        default=np.nan)

estudia = (p["P8586"] == 1) & p["P1088"].notna()
p["ANOS_EDUC"] = np.where(
    estudia,
    anos_estudiante(p["P1088"], p["P1088S1"], p["P6216"]),
    anos_no_estudiante(p["P8587"], p["P8587S1"]))
p["ANOS_EDUC"] = p["ANOS_EDUC"].clip(upper=22)   # 11 años de básica y media + máximo 11 de superior
p["OCUPADO"] = ((p["P6240"] == 1) | (p["P6250"] == 1) | (p["P6260"] == 1) | (p["P6270"] == 1))

m15 = p["P6040"] >= 15
print("Estudiantes con nivel matriculado:", estudia.sum())
print(p[estudia].groupby("P1088")["P1088S1"].agg(["min", "max", "count"]))
print("Personas 15+ sin dato de educación:", p.loc[m15, "ANOS_EDUC"].isna().sum())
print(p.loc[m15, "ANOS_EDUC"].describe())

# 3.2 Jefe de hogar, sexo y edad del jefe
df_hogares = df_hogares.drop(columns=["SEXO_JEFE", "EDAD_JEFE"], errors="ignore")
jefes = (p[p["P6051"] == 1][LLAVE_HOGAR + ["P6020", "P6040"]]
         .drop_duplicates(LLAVE_HOGAR)
         .rename(columns={"P6020": "SEXO_JEFE", "P6040": "EDAD_JEFE"}))
df_hogares = df_hogares.merge(jefes, on=LLAVE_HOGAR, how="left")
print("Hogares sin jefe:", df_hogares["SEXO_JEFE"].isna().sum())

# 3.3 Estrato (vivienda: "Estrato para tarifa"); solo valores 1 a 6 son estratos válidos
df_hogares["ESTRATO"] = df_hogares["P8520S1A1"].where(df_hogares["P8520S1A1"].between(1, 6))
print(df_hogares["ESTRATO"].value_counts(dropna=False).sort_index())

# 3.4 Ingreso del hogar
print(df_hogares[["I_HOGAR", "PERCAPITA"]].describe(percentiles=[.01, .25, .5, .75, .99]))
print("Hogares con I_HOGAR == 0:", round((df_hogares["I_HOGAR"] == 0).mean() * 100, 2), "%")

**Decisiones de limpieza**
- **Años de educación:** quien estudia cuenta los años hasta el grado anterior al que cursa (metodología DANE); el resto usa el último nivel y grado aprobados. Se recorta en 22 años (11 de básica y media + 11 de superior) porque `P6216` traía valores atípicos hasta 31.
- **Jefe de hogar:** `P6051 == 1`. Todos los hogares tienen jefe.
- **Estrato:** solo se aceptan valores de 1 a 6; los códigos 0, 8 y 9 y los vacíos quedan como nulos (11.019 hogares).
- **Ingreso:** el 1,9% de los hogares reporta ingreso 0 y hay valores extremos (máximo de 250 millones por persona). Se conservan en la base; el univariado excluye los ceros y usa escala logarítmica.
- **Códigos especiales:** en `P6090` (afiliación a salud) el código 9 ("no sabe") se trata como no asegurado, como indica la metodología del IPM.

## 4. VARIABLE OBJETIVO: ÍNDICE DE POBREZA MULTIDIMENSIONAL (IPM)
Los microdatos no traen el IPM, así que se reconstruye con la metodología del DANE: 15 indicadores en 5 dimensiones, cada dimensión pesa 0,2 y los indicadores de una misma dimensión pesan igual. Un hogar es pobre si su suma ponderada de privaciones es **≥ 1/3**.

| Dimensión | Indicador | Peso | Variables principales |
|---|---|---|---|
| Educación | I01 Bajo logro educativo (promedio < 9 años, 15+) | 0,10 | `P8587`, `P8587S1`, `P1088`, `P1088S1` |
| Educación | I02 Analfabetismo (15+) | 0,10 | `P6160` |
| Niñez y juventud | I03 Inasistencia escolar (6–16) | 0,05 | `P8586` |
| Niñez y juventud | I04 Rezago escolar (7–17) | 0,05 | `ANOS_EDUC`, `P6040` |
| Niñez y juventud | I05 Barreras de cuidado de primera infancia (0–5) | 0,05 | `P51`, `P6090`, `P8586` |
| Niñez y juventud | I06 Trabajo infantil (12–17) | 0,05 | `P6240`, `P6250`, `P6260`, `P6270` |
| Trabajo | I07 Desempleo de larga duración | 0,10 | `P7250` |
| Trabajo | I08 Empleo informal (sin pensión) | 0,10 | `P6920` |
| Salud | I09 Sin aseguramiento (6+) | 0,10 | `P6090` |
| Salud | I10 Barreras de acceso a salud | 0,10 | `P5665`, `P8563` |
| Vivienda | I11 Sin fuente de agua mejorada | 0,04 | `P8520S5`, `P8530` |
| Vivienda | I12 Inadecuada eliminación de excretas | 0,04 | `P8520S3`, `P8526` |
| Vivienda | I13 Pisos inadecuados | 0,04 | `P4015` |
| Vivienda | I14 Paredes inadecuadas | 0,04 | `P4005` |
| Vivienda | I15 Hacinamiento crítico | 0,04 | `P5010`, `TAM_HOGAR` |

Zona urbana = `CLASE` 1 (cabeceras); rural = `CLASE` 2 (centros poblados y rural disperso).

In [ ]:
# 4. VARIABLE OBJETIVO: IPM (reconstrucción según metodología DANE)
p = df_personas
h = df_hogares
edad = p["P6040"]
ind = h[LLAVE_HOGAR].copy().set_index(LLAVE_HOGAR)

def cuenta(mask):
    """Personas por hogar que cumplen la condición."""
    return p[mask].groupby(LLAVE_HOGAR).size().reindex(ind.index, fill_value=0)

# --- Educación ---
m15 = edad >= 15
n15 = cuenta(m15)
prom = p[m15].groupby(LLAVE_HOGAR)["ANOS_EDUC"].mean().reindex(ind.index)
ind["I01_logro_educ"]    = ((prom < 9) | prom.isna()).astype(int)
ind["I02_analfabetismo"] = ((cuenta(m15 & (p["P6160"] == 2)) > 0) | (n15 == 0)).astype(int)

# --- Niñez y juventud ---
ind["I03_inasistencia"] = (cuenta(edad.between(6, 16) & (p["P8586"] == 2)) > 0).astype(int)
ind["I04_rezago"]       = (cuenta(edad.between(7, 17) & (p["ANOS_EDUC"] < (edad - 6))) > 0).astype(int)

sin_salud = p["P6090"] != 1                      # 2 = no, 9 = no sabe (se cuenta como no asegurado)
priv_0a4  = edad.between(0, 4) & (p["P51"].isin([3, 6, 7]) | sin_salud)
priv_5    = (edad == 5) & (sin_salud | (p["P8586"] == 2))
ind["I05_primera_infancia"] = (cuenta(priv_0a4 | priv_5) > 0).astype(int)
ind["I06_trabajo_infantil"] = (cuenta(edad.between(12, 17) & p["OCUPADO"]) > 0).astype(int)

# --- Trabajo ---
desoc = ~p["OCUPADO"] & p["P7250"].notna()
pea   = p["OCUPADO"] | desoc
dld   = desoc & (p["P7250"] > 52)                # buscando trabajo más de 12 meses
ind["I07_desempleo_larga"] = ((cuenta(dld) > 0) | (cuenta(pea) == 0)).astype(int)

SIN_PEA_PRIVADO_INFORMAL = True                  # si la incidencia sale muy alta, probar False
pea2   = (pea & ~dld) & ~(p["OCUPADO"] & (edad < 18))
formal = p["OCUPADO"] & p["P6920"].isin([1, 3])
n_pea2, n_formal = cuenta(pea2), cuenta(pea2 & formal)
pensionado = cuenta(p["P6920"] == 3) > 0
sin_pea2 = (n_pea2 == 0) & (~pensionado if SIN_PEA_PRIVADO_INFORMAL else False)
ind["I08_informal"] = (((n_pea2 > 0) & (n_formal != n_pea2)) | sin_pea2).astype(int)

# --- Salud ---
ind["I09_sin_aseguramiento"] = (cuenta((edad >= 6) & sin_salud) > 0).astype(int)
ind["I10_barreras_salud"]    = (cuenta((p["P5665"] == 1) & ~p["P8563"].isin([1, 2])) > 0).astype(int)

# --- Vivienda (urbano = CLASE 1, rural = CLASE 2) ---
AGUA_RURAL_PRIV = [4, 5, 6, 8, 9, 10]            # pozo sin bomba, lluvia, río, carro tanque, aguatero, embotellada
urb = h["CLASE"] == 1
ppc = h["TAM_HOGAR"] / h["P5010"].clip(lower=1)  # personas por cuarto para dormir
ind["I11_agua"]         = np.where(urb, h["P8520S5"] != 1, h["P8530"].isin(AGUA_RURAL_PRIV)).astype(int)
ind["I12_excretas"]     = np.where(urb, h["P8520S3"] != 1, h["P8526"].isin([3, 5, 6])).astype(int)
ind["I13_pisos"]        = (h["P4015"] == 7).astype(int).values
ind["I14_paredes"]      = np.where(urb, h["P4005"].isin([5, 7, 8, 9, 10]), h["P4005"].isin([7, 8, 9, 10])).astype(int)
ind["I15_hacinamiento"] = np.where(urb, ppc >= 3, ppc > 3).astype(int)

# --- Índice ponderado ---
PESOS = {"I01_logro_educ": .10, "I02_analfabetismo": .10,
         "I03_inasistencia": .05, "I04_rezago": .05, "I05_primera_infancia": .05, "I06_trabajo_infantil": .05,
         "I07_desempleo_larga": .10, "I08_informal": .10,
         "I09_sin_aseguramiento": .10, "I10_barreras_salud": .10,
         "I11_agua": .04, "I12_excretas": .04, "I13_pisos": .04, "I14_paredes": .04, "I15_hacinamiento": .04}
COLS_IND = list(PESOS)
assert abs(sum(PESOS.values()) - 1) < 1e-9

ind["SCORE_IPM"] = sum(ind[c] * w for c, w in PESOS.items())
ind["POBRE_IPM"] = (ind["SCORE_IPM"] >= 1/3 - 1e-9).astype(int)

df_hogares = df_hogares.drop(columns=COLS_IND + ["SCORE_IPM", "POBRE_IPM"], errors="ignore")
df_hogares = df_hogares.merge(ind.reset_index(), on=LLAVE_HOGAR, how="left")
assert len(df_hogares) == 86405

### 4.1 Validación contra las cifras oficiales del DANE 2023

In [ ]:
# 4.1 Validación (ponderado por FEX_C, a nivel persona)
pp = (df_personas[LLAVE_HOGAR + ["FEX_C"]]
      .merge(df_hogares[LLAVE_HOGAR + ["POBRE_IPM", "SCORE_IPM", "CLASE", "SEXO_JEFE"]], on=LLAVE_HOGAR))
H = lambda d: 100 * np.average(d["POBRE_IPM"], weights=d["FEX_C"])

val = pd.DataFrame({
    "propio": [H(pp), H(pp[pp.CLASE == 1]), H(pp[pp.CLASE == 2]),
               H(pp[pp.SEXO_JEFE == 1]), H(pp[pp.SEXO_JEFE == 2])],
    "DANE_2023": [12.1, 8.3, 25.1, 11.2, 13.2]},
    index=["Nacional", "Cabeceras", "CP y rural disperso", "Jefe hombre", "Jefe mujer"])
print(val.round(1))

pobres = pp[pp.POBRE_IPM == 1]
A = 100 * np.average(pobres["SCORE_IPM"], weights=pobres["FEX_C"])
print(f"\nIntensidad A: {A:.1f}%  (DANE 40.1%)   |   M0 = H*A: {H(pp)/100*A/100:.3f}  (DANE 0.049)")

# Privación por indicador (% de hogares, ponderado)
w = df_hogares["FEX_C"]
print((df_hogares[COLS_IND].apply(lambda c: 100 * np.average(c, weights=w))).round(1))

**Resultado de la validación** (cifras del DANE 2023: boletín técnico de pobreza multidimensional):

| | Propio | DANE 2023 |
|---|---|---|
| Nacional | 11,7% | 12,1% |
| Cabeceras | 7,6% | 8,3% |
| Centros poblados y rural disperso | 24,8% | 25,1% |
| Jefe hombre | 11,0% | 11,2% |
| Jefe mujer | 12,5% | 13,2% |
| Intensidad (A) | 40,2% | 40,1% |
| IPM ajustado (M0) | 0,047 | 0,049 |

La reconstrucción queda a menos de 1 punto porcentual del dato oficial y la intensidad coincide. Las diferencias se explican, probablemente, porque (i) el indicador de primera infancia no incluye el componente de alimentación en el jardín, (ii) el DANE excluye la ruralidad de Amazonía–Orinoquía en sus totales y (iii) hay decisiones propias sobre hogares sin ocupados y pensionados. El indicador de bajo logro educativo sale más alto de lo esperado (≈ 40%) y queda como limitación.

### 4.2 Guardar bases limpias

In [ ]:
# 4.2 Guardar bases limpias
COLS_HOGAR = (LLAVE_HOGAR + ["FEX_C", "CLASE", "REGION", "P1_DEPARTAMENTO", "TAM_HOGAR",
              "I_HOGAR", "PERCAPITA", "ESTRATO", "SEXO_JEFE", "EDAD_JEFE"]
              + COLS_IND + ["SCORE_IPM", "POBRE_IPM"])
COLS_PERS = LLAVE_PERSONA + ["FEX_C", "P6040", "P6020", "P6051", "P6160", "P8586",
                             "P8587", "P8587S1", "ANOS_EDUC", "OCUPADO", "P6090"]

ruta_h = os.path.join(PROCESSED, "ecv2023_hogares_limpia.parquet")
ruta_p = os.path.join(PROCESSED, "ecv2023_personas_limpia.parquet")
df_hogares[COLS_HOGAR].to_parquet(ruta_h, index=False)
df_personas[COLS_PERS].to_parquet(ruta_p, index=False)

for r in (ruta_h, ruta_p):
    print(os.path.basename(r), round(os.path.getsize(r) / 1e6, 1), "MB")

## 5. ANÁLISIS UNIVARIADO
*Estadísticas de la muestra, sin factor de expansión.*

In [ ]:
# 5. ANÁLISIS UNIVARIADO
hog, per = df_hogares, df_personas
ing  = hog.loc[hog["PERCAPITA"] > 0, "PERCAPITA"]          # excluye 1,9% de hogares con ingreso 0
educ = per.loc[per["P6040"] >= 15, "ANOS_EDUC"]

def resumen_uni(s, nombre):
    s = s.dropna()
    q1, q3 = s.quantile([.25, .75])
    return pd.Series({"n": len(s), "media": s.mean(), "mediana": s.median(), "desv_est": s.std(),
                      "min": s.min(), "Q1": q1, "Q3": q3, "IQR": q3 - q1, "max": s.max(),
                      "asimetria": s.skew(), "curtosis": s.kurt()}, name=nombre)

tabla_uni = pd.concat([
    resumen_uni(ing, "Ingreso per cápita (COP)"),
    resumen_uni(np.log10(ing), "log10 ingreso per cápita"),
    resumen_uni(per["P6040"], "Edad (años)"),
    resumen_uni(educ, "Años de educación (15+)"),
    resumen_uni(hog["TAM_HOGAR"], "Tamaño del hogar"),
], axis=1).T
display(tabla_uni)

q1, q3 = ing.quantile([.25, .75]); iqr = q3 - q1
print("Outliers altos de ingreso (regla 1,5·IQR):", round((ing > q3 + 1.5 * iqr).mean() * 100, 1), "%")

In [ ]:
# 5.1 Histogramas
fig, ax = plt.subplots(2, 2, figsize=(13, 9))
sns.histplot(np.log10(ing), bins=50, ax=ax[0, 0]); ax[0, 0].set(title="Ingreso per cápita del hogar (log10 COP)", xlabel="log10(COP)")
sns.histplot(per["P6040"], bins=range(0, 105, 5), ax=ax[0, 1]); ax[0, 1].set(title="Edad", xlabel="años")
sns.histplot(educ.clip(upper=22), bins=range(0, 24), ax=ax[1, 0]); ax[1, 0].set(title="Años de educación (personas de 15+)", xlabel="años")
sns.countplot(x=hog["TAM_HOGAR"].clip(upper=8), ax=ax[1, 1]); ax[1, 1].set(title="Tamaño del hogar (8 = 8 o más)", xlabel="personas")
plt.tight_layout(); plt.savefig(os.path.join(FIGURES, "05_histogramas.png"), dpi=150); plt.show()

# 5.2 Boxplots
fig, ax = plt.subplots(1, 4, figsize=(16, 5))
sns.boxplot(y=np.log10(ing), ax=ax[0]); ax[0].set(title="log10 ingreso per cápita", ylabel="")
sns.boxplot(y=per["P6040"], ax=ax[1]); ax[1].set(title="Edad", ylabel="")
sns.boxplot(y=educ, ax=ax[2]); ax[2].set(title="Años de educación (15+)", ylabel="")
sns.boxplot(y=hog["TAM_HOGAR"], ax=ax[3]); ax[3].set(title="Tamaño del hogar", ylabel="")
plt.tight_layout(); plt.savefig(os.path.join(FIGURES, "05_boxplots.png"), dpi=150); plt.show()

### Conclusiones del análisis univariado
*Estadísticas de la muestra, sin factor de expansión. Las cifras provienen de la tabla `tabla_uni`; verificarlas al volver a ejecutar.*

**Ingreso per cápita del hogar** (n = 84.768; se excluyó el 1,9% de hogares con ingreso 0). La media (≈ $940.000) es cerca de 1,8 veces la mediana (≈ $518.000), con desviación estándar de ≈ $2,06 millones, asimetría de ≈ 42 y curtosis de ≈ 4.000: distribución extremadamente sesgada a la derecha, con valores extremos de hasta $250 millones. El IQR es de $712.500 (Q1 $287.500; Q3 $1.000.000). En escala log10 la distribución es casi simétrica (media 5,72; mediana 5,71; asimetría -0,17), lo que justifica usar el logaritmo en gráficos y modelos. El boxplot muestra además una cola baja de ingresos muy pequeños (mínimo de ≈ $21 por persona), que se conserva y se declara como limitación.

**Edad** (n = 240.212). Media 34,7 y mediana 33 años, con rango de 0 a 104. IQR de 36 años (Q1 16; Q3 52), asimetría de 0,35 y curtosis de -0,83. Es una población joven: la mayor concentración está entre los 0 y 20 años, con cola decreciente hacia edades avanzadas.

**Años de educación, personas de 15 años o más** (n = 184.563; recortado en 22 años). Media ≈ 8,1 y mediana 9 años (básica secundaria). El 25% tiene 5 años o menos y el 75% tiene 11 años o menos. Distribución casi simétrica y aplanada, con picos en 5 y 11 años, que corresponden a primaria y bachillerato completos.

**Tamaño del hogar** (n = 86.405). Media 2,78 y mediana 3 personas, con moda en 2. IQR de 2 (Q1 2; Q3 4), asimetría de 1,10 y curtosis de 2,51: la mayoría de los hogares tiene entre 2 y 4 personas y los hogares grandes son una minoría (máximo 24).

## Estado y pendientes
**Listo:** secciones 0 a 5. Bases limpias en `data/processed/` (hogares y personas, en parquet).

**IPM:** reconstrucción propia validada contra el DANE 2023 (incidencia nacional 11,7% vs 12,1%; intensidad 40,2% vs 40,1%).

**Limitaciones conocidas:**
- Primera infancia (I05) sin el componente de alimentación en el jardín.
- Bajo logro educativo (I01) sale ≈ 40%, más alto de lo esperado.
- 11.019 hogares sin estrato válido.
- 1,9% de hogares con ingreso 0, excluidos del univariado.
- El univariado es de muestra, sin factor de expansión.

**Siguiente (Juan):** cargar los parquet de `data/processed/` (sin repetir la limpieza), hacer la Sección 6 (multivariado con `POBRE_IPM`, `CLASE`, `REGION`, `SEXO_JEFE`, `ESTRATO`), la Sección 7 (3 hallazgos), la 8 (conclusiones y siguientes pasos) y la ficha resumen.

## 6. ANÁLISIS MULTIVARIADO
*(pendiente: Juan)*

In [ ]:
# 6. MULTIVARIADO: cargar las bases limpias (no hace falta repetir la limpieza)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
PROCESSED = os.path.join(BASE_PATH, "data", "processed")
FIGURES   = os.path.join(BASE_PATH, "figures")
sns.set_theme(style="whitegrid", context="notebook")

hog = pd.read_parquet(os.path.join(PROCESSED, "ecv2023_hogares_limpia.parquet"))
per = pd.read_parquet(os.path.join(PROCESSED, "ecv2023_personas_limpia.parquet"))
print(hog.shape, per.shape)
hog.head()

## 7. HALLAZGOS
*(pendiente: Juan)*

## 8. CONCLUSIONES Y SIGUIENTES PASOS
*(pendiente: Juan)*